In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [4]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.5",
    input="Can you tell me the weather in hyderabad today?"
)

print(response.output_text)

I can’t access live weather data from here, so I can’t give the exact weather in Hyderabad today.

If you mean **Hyderabad, India**, check a live source such as:
- Google Weather
- IMD website/app
- AccuWeather
- Weather.com

In mid-June, Hyderabad is typically **warm and humid**, with a chance of **monsoon showers or thunderstorms**.


In [ ]:
import requests

def get_weather(city: str) -> dict:
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()

    if not geo.get("results"):
        return {"error": f"could not find city {city}"}

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current_weather": True,
        },
    ).json()

    return wx["current_weather"]


get_weather("Hyderabad")


{'time': '2026-06-14T16:00',
 'interval': 900,
 'temperature': 28.1,
 'windspeed': 7.1,
 'winddirection': 240,
 'is_day': 0,
 'weathercode': 0}

In [9]:
## define a tool

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. Tokyo",
                }
            },
            "required": ["city"],
        },
    }
]



response = client.responses.create(
    model="gpt-5.5",
    input="Can you tell me the weather in hyderabad today?", 
    tools = tools
)

# print(response.output_text)
import json
for item in response.output:
    if item.type == "function_call":
        print(f"Model wants to call: {item.name}")
        args = json.loads(item.arguments)

        result = get_weather(**args) # i am calling the tool

        ## send the result back to the LLM
        final_response = client.responses.create(
            model="gpt-5.5",
            previous_response_id=response.id,
            input=[
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(result),
                }
            ],
        )
final_response


Model wants to call: get_weather


Response(id='resp_073f25f2ac51ec88006a2ed2e357cc81948197a4568678b65b', created_at=1781453539.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.5-2026-04-23', object='response', output=[ResponseReasoningItem(id='rs_073f25f2ac51ec88006a2ed2e4e410819483e19b5cd4e2d58b', summary=[], type='reasoning', content=[], encrypted_content=None, status=None), ResponseOutputMessage(id='msg_073f25f2ac51ec88006a2ed2e5a5f08194b200de58563f138e', content=[ResponseOutputText(annotations=[], text='The weather in Hyderabad today is **clear**, with a temperature of about **28°C**.  \n\nWind is light at around **7 km/h**, blowing from the **southwest**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781453542.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id

In [11]:
final_response.output_text



'The weather in Hyderabad today is **clear**, with a temperature of about **28°C**.  \n\nWind is light at around **7 km/h**, blowing from the **southwest**.'

In [8]:
response.output

[ResponseFunctionToolCall(arguments='{"city":"Hyderabad"}', call_id='call_5XoxP7tzFCYiDPDwWI5h4z2C', name='get_weather', type='function_call', id='fc_0f8de435fb3912ac006a2ed1a2a54c8196be91924626916f89', status='completed')]

{'base': 'USD', 'target': 'INR', 'rate': 95.12}


---
## Part 1: Tool Calling with the OpenAI Responses API

Tool calling always follows the same three-step pattern regardless of the provider:
1. **Define tools** as JSON schemas and pass them to the model.
2. **Detect tool calls** in the response and execute the functions yourself.
3. **Send results back** so the model can produce a final answer.

The tricky part is that every provider implements those three steps differently. Here's how OpenAI does it:

- **`"parameters"`** — key used inside a tool definition to describe inputs.
- **`response.output`** — list to iterate; check `item.type == "function_call"` to find tool calls.
- **`item.call_id`** — ID you must echo back when returning a result.
- **`"function_call_output"`** — input type for returning tool results.
- **`previous_response_id`** — threads the follow-up call to the original so you don't manually pass history.


In [ ]:
def get_weather(city: str) -> dict:
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()

    if not geo.get("results"):
        return {"error": f"could not find city {city}"}

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current_weather": True,
        },
    ).json()

    return wx["current_weather"]



def get_exchange_rate(base: str, target: str) -> dict:
    import httpx
    r = httpx.get(
        "https://api.frankfurter.dev/v1/latest",
        params={"from": base.upper(), "to": target.upper()},
    )
    data = r.json()
    rate = data["rates"][target.upper()]
    return {"base": base.upper(), "target": target.upper(), "rate": rate}

get_exchange_rate("USD", "INR")


tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Tokyo"}
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "Get the latest exchange rate from one currency to another",
        "parameters": {
            "type": "object",
            "properties": {
                "base":   {"type": "string", "description": "3-letter ISO code to convert from, e.g. USD"},
                "target": {"type": "string", "description": "3-letter ISO code to convert to, e.g. INR"},
            },
            "required": ["base", "target"],
        },
    },
]

response = client.responses.create(
    model="gpt-5.5",
    input="What's the weather in Hyderabad, and what's the USD to INR rate?",
    tools=tools,
)



available_tools = {
    "get_weather": get_weather,
    "get_exchange_rate": get_exchange_rate,
}

tool_outputs = []
for item in response.output:
    if item.type == "function_call":
        print(f"Model wants to call: {item.name} with {item.arguments}")
        args = json.loads(item.arguments)
        result = available_tools[item.name](**args)   # dispatch by name
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result),
        })


final_response = client.responses.create(
    model="gpt-5.5",
    previous_response_id=response.id,
    input=tool_outputs,
)

print(final_response.output_text)



Model wants to call: get_weather with {"city":"Hyderabad"}
Model wants to call: get_exchange_rate with {"base":"USD","target":"INR"}
Hyderabad weather: 27.7°C, clear conditions, wind 7.3 km/h from the WSW.  

USD to INR: 1 USD = 95.12 INR.


In [17]:
response.output

[ResponseFunctionToolCall(arguments='{"city":"Hyderabad"}', call_id='call_581xKOxmv8MtbzL3J72sZCac', name='get_weather', type='function_call', id='fc_083703b216bfa7cc006a2ed71397e88194b82c0cba248fb6c9', status='completed'),
 ResponseFunctionToolCall(arguments='{"base":"USD","target":"INR"}', call_id='call_iGz6cTRd7l6Ym3wqvxhYtIU7', name='get_exchange_rate', type='function_call', id='fc_083703b216bfa7cc006a2ed71398008194b20a603da5b02d51', status='completed')]


---
## Part 2: Same Tools with the Anthropic Claude API

Every LLM provider has its own tool-calling API. Compare with the OpenAI code above:

| Concept | OpenAI | Anthropic |
|---|---|---|
| Tool schema key | `"parameters"` | `"input_schema"` |
| Detect tool calls | `item.type == "function_call"` in `response.output` | `block.type == "tool_use"` in `response.content` |
| Tool call ID | `item.call_id` | `block.id` |
| Return results | `function_call_output` item + `previous_response_id` | `tool_result` block in a new **user** message |
| History threading | Automatic via `previous_response_id` | Manual — must pass full `messages` list each call |


In [20]:
# !pip install anthropic

In [23]:
import os
os.getenv("OPENROUTER_API_KEY")

'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'

In [ ]:

import anthropic
import os

client_claude = anthropic.Anthropic()

# KEY DIFFERENCE #1: Anthropic uses "input_schema" instead of "parameters"
tools_claude = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "input_schema": {                          # <-- "input_schema", not "parameters"
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Tokyo"}
            },
            "required": ["city"],
        },
    },
    {
        "name": "get_exchange_rate",
        "description": "Get the latest exchange rate from one currency to another",
        "input_schema": {
            "type": "object",
            "properties": {
                "base":   {"type": "string", "description": "3-letter ISO code to convert from, e.g. USD"},
                "target": {"type": "string", "description": "3-letter ISO code to convert to, e.g. INR"},
            },
            "required": ["base", "target"],
        },
    },
]

response = client_claude.messages.create(
    model="anthropic/claude-3-haiku",
    max_tokens=1024,
    messages=[{"role": "user", "content": "What's the weather in Hyderabad, and what's the USD to INR rate?"}],
    tools=tools_claude,
)

print(f"Stop reason: {response.stop_reason}")   # "tool_use" when the model calls tools


NotFoundError: <!DOCTYPE html><html data-dpl-id="dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1, minimum-scale=1"/><link rel="stylesheet" href="/_next/static/chunks/0puvfppvifekr.css?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" data-precedence="next"/><link rel="stylesheet" href="/_next/static/chunks/0-n3gli~3sck4.css?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" data-precedence="next"/><link rel="stylesheet" href="/_next/static/chunks/03.w9u7.gwils.css?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" data-precedence="next"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/13esdo-g0nv2m.js?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir"/><script src="/_next/static/chunks/0dxpkqx83t-ub.js?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" async=""></script><script src="/_next/static/chunks/0njw1x-hs8jxp.js?dpl=dpl_C4aTBhmehL9iVe1X2arV1UFjnYir" async=""></script><script src="/_next/static/chunks/0gs9o4a4wh09l.js?dp

In [ ]:

# KEY DIFFERENCE #2: Anthropic blocks have type "tool_use" (not a separate output list)
tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

tool_results = []
for block in tool_use_blocks:
    print(f"Model wants to call: {block.name} with {block.input}")
    result = available_tools[block.name](**block.input)
    tool_results.append({
        "type": "tool_result",
        "tool_use_id": block.id,       # Matches block.id, not call_id
        "content": json.dumps(result),
    })

# KEY DIFFERENCE #3: No previous_response_id — must manually pass the full message history
final_response = client_claude.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What's the weather in Hyderabad, and what's the USD to INR rate?"},
        {"role": "assistant", "content": response.content},  # Full response incl. tool_use blocks
        {"role": "user",      "content": tool_results},      # Tool results as a new user turn
    ],
    tools=tools_claude,
)

print(final_response.content[0].text)



---
## Part 3: Same Tools with the Google Gemini API

Key differences from OpenAI and Anthropic:
- **Tool schema**: Uses `FunctionDeclaration` objects inside a `Tool` wrapper — not a plain dict.
- **Type names**: `"OBJECT"`, `"STRING"` (uppercase), vs. `"object"`, `"string"` in OpenAI/Anthropic.
- **Response**: Function calls come back as `function_call` parts inside `response.candidates[0].content.parts`.
- **Tool results**: Sent back as `function_response` parts using `Part.from_function_response()`.
- **History threading**: Same manual approach as Anthropic — must pass the full `contents` list each turn.


In [30]:
# !pip install google-genai

In [47]:

from google import genai
from google.genai import types as genai_types

client_gemini = genai.Client()  # reads GEMINI_API_KEY from env

# KEY DIFFERENCE: Gemini wraps declarations in a Tool object
# Schema uses a dedicated Schema class with UPPERCASE type names
gemini_tools = genai_types.Tool(function_declarations=[
    genai_types.FunctionDeclaration(
        name="get_weather",
        description="Get the current weather for a city",
        parameters=genai_types.Schema(
            type="OBJECT",         # Uppercase, not "object"
            properties={
                "city": genai_types.Schema(type="STRING", description="City name, e.g. Tokyo"),
            },
            required=["city"],
        ),
    ),
    genai_types.FunctionDeclaration(
        name="get_exchange_rate",
        description="Get the latest exchange rate from one currency to another",
        parameters=genai_types.Schema(
            type="OBJECT",
            properties={
                "base":   genai_types.Schema(type="STRING", description="3-letter ISO code to convert from"),
                "target": genai_types.Schema(type="STRING", description="3-letter ISO code to convert to"),
            },
            required=["base", "target"],
        ),
    ),
])

response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents="What's the weather in Hyderabad today, and what's the USD to INR rate today?",
)
print("*"*10)
print( "Without tool call: ", response)
print("*"*10)


print( "With tool call: ", )
response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents="What's the weather in Hyderabad, and what's the USD to INR rate?",
    config=genai_types.GenerateContentConfig(tools=[gemini_tools]),
)

print(response)

print(f"Finish reason: {response.candidates[0].finish_reason}")


**********
Without tool call:  sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Here's the information you requested:

**Weather in Hyderabad Today (as of late morning/early afternoon, June 14, 2024):**

*   **Temperature:** Around **31°C (88°F)**
*   **Conditions:** Mostly **sunny with some scattered clouds**.
*   **Feels Like:** Due to humidity, it feels warmer, closer to **35°C (95°F)**.
*   **Humidity:** Approximately 60-70%.
*   **Wind:** Light breeze.

**USD to INR Exchange Rate Today (as of June 14, 2024):**

*   **1 US Dollar (USD) is approximately equal to 83.47 Indian Rupees (INR).**

Please note that currency exchange rates can fluctuate throughout the day based on market conditions."""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='GQAxaq-eDousjuMP2o2dmA

In [ ]:
response

In [48]:

# Collect function calls from response parts
function_calls = []
for part in response.candidates[0].content.parts:
    if part.function_call:
        fc = part.function_call
        print(f"Model wants to call: {fc.name} with {dict(fc.args)}")
        function_calls.append(fc)

# Execute tools and build function_response parts
function_responses = []
for fc in function_calls:
    result = available_tools[fc.name](**dict(fc.args))
    function_responses.append(
        genai_types.Part.from_function_response(name=fc.name, response=result)
    )

# KEY DIFFERENCE: Must manually thread the full content history
final_response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        genai_types.Content(role="user", parts=[genai_types.Part(text="What's the weather in Hyderabad, and what's the USD to INR rate?")]),
        response.candidates[0].content,                              # Model turn with function_call parts
        genai_types.Content(role="user", parts=function_responses),  # Function results
    ],
    config=genai_types.GenerateContentConfig(tools=[gemini_tools]),
)

print(final_response.text)


Model wants to call: get_weather with {'city': 'Hyderabad'}
Model wants to call: get_exchange_rate with {'target': 'INR', 'base': 'USD'}
The weather in Hyderabad is clear with a temperature of 34.9°C. The current USD to INR exchange rate is 1 USD = 94.72 INR.


In [52]:
# get_weather("Hyderabad")
get_exchange_rate("USD", "INR")

{'base': 'USD', 'target': 'INR', 'rate': 94.72}


---
## Part 4: Same Tools with LangChain (Unified Abstraction)

Every provider uses a different schema format, a different message type to return tool results, and a different way to check whether tools were called. **LangChain abstracts all of that away.**

Key ideas:
- **`@tool` decorator** — auto-generates the JSON schema from Python type hints and the docstring; no manual dict needed.
- **`.bind_tools(tools)`** — attaches tools to any `ChatModel` using that provider's native format under the hood.
- **`ToolMessage`** — a single message type for tool results, regardless of provider.
- **Swap the LLM in one line** — `ChatOpenAI`, `ChatAnthropic`, `ChatGoogleGenerativeAI` all share the same interface, so the dispatch loop never changes.


In [4]:

from langchain_core.tools import tool
import requests, httpx

# The @tool decorator generates the JSON schema from type hints + docstring automatically
@tool
def get_weather_lc(city: str) -> dict:
    """Get the current weather for a city."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()
    if not geo.get("results"):
        return {"error": f"could not find city {city}"}
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
    ).json()
    return wx["current_weather"]

@tool
def get_exchange_rate_lc(base: str, target: str) -> dict:
    """Get the latest exchange rate from one currency to another."""
    r = httpx.get(
        "https://api.frankfurter.dev/v1/latest",
        params={"from": base.upper(), "to": target.upper()},
    )
    data = r.json()
    rate = data["rates"][target.upper()]
    return {"base": base.upper(), "target": target.upper(), "rate": rate}

lc_tools = [get_weather_lc, get_exchange_rate_lc]
print("LangChain tools:", [t.name for t in lc_tools])


LangChain tools: ['get_weather_lc', 'get_exchange_rate_lc']


In [5]:

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

def run_with_tools(llm, question: str) -> str:
    """Works identically regardless of which LangChain LLM you pass in."""
    llm_with_tools = llm.bind_tools(lc_tools)

    messages = [HumanMessage(question)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # Same tool dispatch loop — no provider-specific branching needed
    tool_map = {t.name: t for t in lc_tools}
    for tool_call in response.tool_calls:
        print(f"Calling: {tool_call['name']} with {tool_call['args']}")
        result = tool_map[tool_call["name"]].invoke(tool_call)
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    final = llm_with_tools.invoke(messages)
    return final.content

# With OpenAI
llm_openai = ChatOpenAI(model="gpt-4o")
result = run_with_tools(llm_openai, "What's the weather in Hyderabad, and what's the USD to INR rate?")
print("=== OpenAI ===")
print(result)


Calling: get_weather_lc with {'city': 'Hyderabad'}
Calling: get_exchange_rate_lc with {'base': 'USD', 'target': 'INR'}
=== OpenAI ===
The current weather in Hyderabad is as follows:

- Temperature: 34.9°C
- Windspeed: 11.0 km/h
- Wind direction: 310° (Northwest)
- It's daytime, and the sky is clear with no significant weather codes.

The latest exchange rate from USD to INR is 94.72.


In [56]:

# !pip install --upgrade langchain-core langchain-anthropic langchain-google-genai


In [7]:
# from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

# With Anthropic — change JUST the LLM, same run_with_tools function!
# llm_claude = ChatAnthropic(model="claude-opus-4-8")
# result = run_with_tools(llm_claude, "What's the weather in Hyderabad, and what's the USD to INR rate?")
# print("=== Anthropic Claude ===")
# print(result)

# print()

# With Gemini — change JUST the LLM, same run_with_tools function!
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
result = run_with_tools(llm_gemini, "What's the weather in Hyderabad, and what's the USD to INR rate?")
print("=== Google Gemini ===")
print(result)


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 47.062166223s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url